In [2]:
library(Seurat)
library(Seurat)
library(data.table)
library(RColorBrewer)
library(ggplot2)
library(cowplot)
library(Seurat)
library(Rcpp)
library(harmony)
library(stringr)
library(dplyr)
library(patchwork)
library(ggplot2)
library(ggpubr)
library(reshape2)
library(ggridges)
library(stringr)

Attaching SeuratObject


载入程辑包：‘dplyr’


The following objects are masked from ‘package:data.table’:

    between, first, last


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



载入程辑包：‘patchwork’


The following object is masked from ‘package:cowplot’:

    align_plots



载入程辑包：‘ggpubr’


The following object is masked from ‘package:cowplot’:

    get_legend



载入程辑包：‘reshape2’


The following objects are masked from ‘package:data.table’:

    dcast, melt




In [3]:
drds <- "/DATA/User/liqian7/pienapple/00AS_project/82plot_figure/42area_bulk/00cellsegmentation/"

In [4]:
color <- c("Macrophage"='#df65b0',"Modulated_SMC"="#A1CFFA",
           "SMC"='#1f78b4',"EC"='#238b45',"Fibroblast"="#8b4a4b","Immune"='#ce1256')

In [19]:
order <- c("Macrophage","Modulated_SMC","SMC","EC","Fibroblast")

In [40]:
i=0
for(s in Sys.glob(file.path(drds,"*.RDS"))){
    i=i+1
    key <- gsub(".RDS","",basename(s))
    sp <- readRDS(paste0(drds,key,".RDS"))
    n=0
    for(k in unique(sp@meta.data$first_type)){
        n=n+1
        pbmc <- subset(sp,subset = first_type==k)
        counts <- as.data.frame(rowSums(as.matrix(pbmc@assays$RNA@counts)))
        colnames(counts) <- paste0(key,"_",k)
        if(n==1){
            m <- counts
        }
        else{
            m <- merge(m,counts,all=TRUE,by=0)
            rownames(m) <- m$Row.names
            m <- m[,-1]
        }
    }
    if(i==1){
        mer <- m
    }
    else{
        mer <- merge(mer,m,all=TRUE,by=0)
        rownames(mer) <- mer$Row.names
        mer <- mer[,-1]
    }
}

In [43]:
mer1 <- replace(mer, is.na(mer), 0)

In [45]:
#创建seurat对象
seurat<-CreateSeuratObject(counts =mer1, 
                       project = "SeuratObject")

In [107]:
type <- read.csv("/DATA/User/liqian7/pienapple/00AS_project/82plot_figure/29gsva_percent/cellsegmentation_AD_length_score.csv",row.names = 1)

In [110]:
sp@meta.data$stage <- rownames(sp@meta.data)
sp@meta.data$AD <- rownames(sp@meta.data)
sp@meta.data$cell <- rownames(sp@meta.data)
sp@meta.data$ID <- rownames(sp@meta.data)

In [ ]:
c("_Macrophage","_EC","_SMC","_Modulated_SMC","_Immune","_Fibroblast")

In [121]:
sp@meta.data$ID <- str_replace(sp@meta.data$ID,"_Immune","")

In [123]:
length(unique(sp@meta.data$ID))

[1] 162

In [124]:
for(i in unique(sp@meta.data$ID)){
    AD <- type[which(rownames(type)==i),]$AD
    stage <- type[which(rownames(type)==i),]$stage
    sp@meta.data[which(sp@meta.data$ID==i),]$AD <- AD
    sp@meta.data[which(sp@meta.data$ID==i),]$stage <- stage
    sp@meta.data$cell <- str_replace(sp@meta.data$cell,paste0(i,"_"),"")
}

In [125]:
sp@meta.data$stage_cell <- paste0(sp@meta.data$stage,"_",sp@meta.data$cell)

In [126]:
sp@meta.data$stage_cell <- str_replace(sp@meta.data$stage_cell,"plaque_","")

In [131]:
sp@meta.data

,orig.ident,nCount_RNA,nFeature_RNA,stage,AD,cell,ID,stage_cell
,<fct>,<dbl>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>
SS200000325TL_D5v1_above_division1_Macrophage,SS200000325TL,86870,9160,plaque_stage3,MAW1,Macrophage,SS200000325TL_D5v1_above_division1,stage3_MAC
SS200000325TL_D5v1_above_division1_Fibroblast,SS200000325TL,57949,7981,plaque_stage3,MAW1,Fibroblast,SS200000325TL_D5v1_above_division1,stage3_FIB
SS200000325TL_D5v1_above_division1_Modulated_SMC,SS200000325TL,75026,8612,plaque_stage3,MAW1,Modulated_SMC,SS200000325TL_D5v1_above_division1,stage3_MSM
SS200000325TL_D5v1_above_division1_SMC,SS200000325TL,18119,4514,plaque_stage3,MAW1,SMC,SS200000325TL_D5v1_above_division1,stage3_SMC
SS200000325TL_D5v1_above_division1_EC,SS200000325TL,2365,1124,plaque_stage3,MAW1,EC,SS200000325TL_D5v1_above_division1,stage3_EC
SS200000325TL_D5v1_above_division2_Macrophage,SS200000325TL,120663,9936,plaque_stage4,MAW2,Macrophage,SS200000325TL_D5v1_above_division2,stage4_MAC
SS200000325TL_D5v1_above_division2_SMC,SS200000325TL,20997,4973,plaque_stage4,MAW2,SMC,SS200000325TL_D5v1_above_division2,stage4_SMC
SS200000325TL_D5v1_above_division2_Modulated_SMC,SS200000325TL,75480,8739,plaque_stage4,MAW2,Modulated_SMC,SS200000325TL_D5v1_above_division2,stage4_MSM
SS200000325TL_D5v1_above_division2_Fibroblast,SS200000325TL,22577,5318,plaque_stage4,MAW2,Fibroblast,SS200000325TL_D5v1_above_division2,stage4_FIB


In [128]:
sp@meta.data$stage_cell <- str_replace(sp@meta.data$stage_cell,"Macrophage","MAC")
sp@meta.data$stage_cell <- str_replace(sp@meta.data$stage_cell,"Modulated_SMC","MSM")
sp@meta.data$stage_cell <- str_replace(sp@meta.data$stage_cell,"Immune","IMM")
sp@meta.data$stage_cell <- str_replace(sp@meta.data$stage_cell,"Fibroblast","FIB")

In [11]:
saveRDS(sp,"stage_bulk_celltype.RDS")

In [1]:
sp <- readRDS("/DATA/User/liqian7/pienapple/00AS_project/82plot_figure/46bulk_stage_allcell/stage_bulk_celltype.RDS")

In [139]:
#####FIB:8000
#####SMC:10000
#####MSMC:8000
#####EC:5000
####Immune:filter
####Macrophage:8000

In [149]:
sp@meta.data$filter <- "filter"

In [6]:
sp@meta.data[which(sp@meta.data$cell=="Fibroblast"&sp@meta.data$nCount_RNA>8000),]$filter <- "save"
sp@meta.data[which(sp@meta.data$cell=="SMC"&sp@meta.data$nCount_RNA>10000),]$filter <- "save"
sp@meta.data[which(sp@meta.data$cell=="Modulated_SMC"&sp@meta.data$nCount_RNA>8000),]$filter <- "save"
sp@meta.data[which(sp@meta.data$cell=="EC"&sp@meta.data$nCount_RNA>5000),]$filter <- "save"
sp@meta.data[which(sp@meta.data$cell=="Macrophage"&sp@meta.data$nCount_RNA>8000),]$filter <- "save"

In [7]:
table(sp@meta.data$filter)


filter   save 
   228    612 

In [ ]:
sp <- NormalizeData(sp,normalization.method = "LogNormalize",scale.factor = 100000)

In [8]:
sp1 <- subset(sp,subset = filter=="save")

In [9]:
saveRDS(sp1,"satge_bulk_celltype_filter.RDS")

In [ ]:
exp <- t(as.data.frame(sp1@assays$RNA@data))
mer <- merge(sp1@meta.data,exp,all = TRUE,by = 0)
rownames(mer) <- mer$Row.names
m <- mer[,-1]

In [10]:
####creat_scenic_data

In [12]:
setwd("/DATA/User/liqian7/pienapple/00AS_project/82plot_figure/46bulk_stage_allcell/00scenic/")

In [14]:
write.csv(t(as.matrix(sp1@assays$RNA@counts)),file = "expr.csv")